# Teste Técnico Python

## Autor: Maurício Marques

### Descrição
Este notebook contém soluções para três desafios:
1. Função dinâmica para consulta ao banco de dados
2. Visualização de Ticket Médio por loja e categoria
3. Análise exploratória de filmes IMDB

## 1. Setup e Configuração

### 1.1 Instalação de Dependências
```bash
pip install sqlalchemy pymysql pandas plotly
```

In [1]:
# Importação de bibliotecas
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
from typing import List, Optional
import warnings

warnings.filterwarnings('ignore')

In [2]:
# Configuração do banco de dados
class DatabaseConfig:
    """Classe para gerenciar configurações de banco de dados."""
    
    HOST = "35.199.115.174"
    PORT = 3306
    USER = "looqbox-challenge"
    PASSWORD = "looq-challenge"
    DB_NAME = "looqbox-challenge"
    
    @classmethod
    def get_connection_string(cls) -> str:
        """Retorna string de conexão MySQL."""
        return f"mysql+pymysql://{cls.USER}:{cls.PASSWORD}@{cls.HOST}:{cls.PORT}/{cls.DB_NAME}"
    
    @classmethod
    def create_engine(cls):
        """Cria e retorna engine SQLAlchemy."""
        return create_engine(cls.get_connection_string())


# Criação do engine global
engine = DatabaseConfig.create_engine()
print("✓ Conexão com banco de dados estabelecida")

✓ Conexão com banco de dados estabelecida


## 2. Case 1: Função Dinâmica de Consulta

### Objetivo
Criar uma função flexível para consultar dados de vendas por produto, loja e datas.

### Requisitos
- Parâmetros: `product_code` (int), `store_code` (int), `dates` (list)
- Retornar todas as colunas da tabela `data_product_sales`
- Ser utilizável por outras equipes

In [3]:
def retrieve_data(
    product_code: int,
    store_code: int,
    dates: List[str],
    engine=engine
) -> pd.DataFrame:
    """
    Consulta dados de vendas de produtos filtrados por código de produto,
    código de loja e lista de datas.
    
    Parameters
    ----------
    product_code : int
        Código do produto a ser consultado
    store_code : int
        Código da loja a ser consultada
    dates : List[str]
        Lista de datas no formato ISO (YYYY-MM-DD)
    engine : sqlalchemy.engine.Engine, optional
        Engine de conexão com banco de dados
    
    Returns
    -------
    pd.DataFrame
        DataFrame com os dados filtrados
    
    Examples
    --------
    >>> data = retrieve_data(18, 1, ['2019-01-01', '2019-01-02'])
    >>> print(data.shape)
    (2, 5)
    
    Raises
    ------
    ValueError
        Se os parâmetros não forem válidos
    sqlalchemy.exc.SQLAlchemyError
        Se houver erro na consulta ao banco
    """
    # Validação de entrada
    if not isinstance(product_code, int) or product_code <= 0:
        raise ValueError("product_code deve ser um inteiro positivo")
    
    if not isinstance(store_code, int) or store_code <= 0:
        raise ValueError("store_code deve ser um inteiro positivo")
    
    if not dates or not isinstance(dates, list):
        raise ValueError("dates deve ser uma lista não vazia")
    
    # Query parametrizada (proteção contra SQL injection)
    query = text("""
        SELECT 
            store_code,
            product_code,
            date,
            sales_value,
            sales_qty
        FROM data_product_sales
        WHERE product_code = :product_code
          AND store_code = :store_code
    """)
    
    try:
        # Execução da query
        df = pd.read_sql_query(
            query,
            engine,
            params={"product_code": product_code, "store_code": store_code}
        )
        
        # Conversão e filtragem de datas
        df['date'] = pd.to_datetime(df['date'])
        dates_dt = pd.to_datetime(dates)
        df_filtered = df[df['date'].isin(dates_dt)].reset_index(drop=True)
        
        return df_filtered
    
    except Exception as e:
        print(f"Erro ao consultar dados: {e}")
        raise

In [4]:
# Teste da função
print("Testando função retrieve_data...\n")

result = retrieve_data(
    product_code=18,
    store_code=1,
    dates=['2019-01-01', '2019-01-02']
)

print(f"Registros encontrados: {len(result)}")
print("\nAmostra dos dados:")
display(result)

Testando função retrieve_data...

Registros encontrados: 2

Amostra dos dados:


,store_code,product_code,date,sales_value,sales_qty
0,1,18,2019-01-01,708.5,65.0
1,1,18,2019-01-02,1297.1,119.0


## 3. Case 2: Visualização de Ticket Médio

### Objetivo
Criar visualização do Ticket Médio por loja e categoria de negócio no período de out-dez/2019.

### Requisitos
- Usar queries fornecidas sem modificação
- Filtrar período: 2019-10-01 a 2019-12-31
- Calcular Ticket Médio (TM = Valor / Quantidade)

In [5]:
# Query 1: Dados cadastrais das lojas
QUERY_STORE_CAD = """
SELECT
    STORE_CODE,
    STORE_NAME,
    START_DATE,
    END_DATE,
    BUSINESS_NAME,
    BUSINESS_CODE
FROM data_store_cad
"""

# Query 2: Dados de vendas
QUERY_STORE_SALES = """
SELECT
    STORE_CODE,
    DATE,
    SALES_VALUE,
    SALES_QTY
FROM data_store_sales
WHERE DATE BETWEEN '2019-01-01' AND '2019-12-31'
"""

print("✓ Queries definidas")

✓ Queries definidas


In [6]:
def process_ticket_medio(
    start_date: str = '2019-10-01',
    end_date: str = '2019-12-31'
) -> pd.DataFrame:
    """
    Processa e calcula o ticket médio por loja e categoria.
    
    Parameters
    ----------
    start_date : str
        Data inicial do período (formato YYYY-MM-DD)
    end_date : str
        Data final do período (formato YYYY-MM-DD)
    
    Returns
    -------
    pd.DataFrame
        DataFrame com colunas: Loja, Categoria, TM
    """
    # Carregamento dos dados
    print("Carregando dados...")
    df_stores = pd.read_sql_query(QUERY_STORE_CAD, engine)
    df_sales = pd.read_sql_query(QUERY_STORE_SALES, engine)
    
    # Merge dos dados
    df_merged = pd.merge(
        df_stores,
        df_sales,
        on='STORE_CODE',
        how='inner'
    )
    
    # Conversão e filtro de datas
    df_merged['DATE'] = pd.to_datetime(df_merged['DATE'])
    mask = df_merged['DATE'].between(start_date, end_date)
    df_filtered = df_merged[mask].copy()
    
    # Cálculo do Ticket Médio
    df_filtered['TM'] = df_filtered['SALES_VALUE'] / df_filtered['SALES_QTY']
    
    # Agregação por loja e categoria
    df_result = (
        df_filtered
        .groupby(['STORE_NAME', 'BUSINESS_NAME'], as_index=False)
        .agg({'TM': 'mean'})
        .rename(columns={
            'STORE_NAME': 'Loja',
            'BUSINESS_NAME': 'Categoria'
        })
    )
    
    # Arredondamento
    df_result['TM'] = df_result['TM'].round(2)
    
    print(f"✓ Processamento concluído: {len(df_result)} lojas")
    return df_result.sort_values('Loja').reset_index(drop=True)


# Execução do processamento
df_ticket_medio = process_ticket_medio()
display(df_ticket_medio)

Carregando dados...
✓ Processamento concluído: 20 lojas


,Loja,Categoria,TM
0,Bahia,Atacado,15.39
1,Bangkok,Posto,13.67
2,Belem,Proximidade,15.37
3,Berlin,Proximidade,15.39
4,Buenos Aires,Atacado,15.39
5,Chicago,Varejo,15.53
6,Dubai,Atacado,15.39
7,Hong Kong,Farma,26.35
8,London,Farma,28.99
9,Madri,Farma,29.03


## 4. Case 3: Análise Exploratória de Filmes IMDB

### Objetivo
Criar visualização interessante usando dados da tabela IMDB_movies.

### Escolha da Visualização
**Gráfico de Dispersão: Avaliação vs Receita**

#### Justificativa:
1. **Insights de Negócio**: Identifica filmes com discrepância entre crítica e sucesso comercial
2. **Múltiplas Dimensões**: Visualiza simultaneamente rating, receita, popularidade (votos) e metadados
3. **Interatividade**: Plotly permite exploração detalhada dos dados
4. **Análise Estratégica**: Útil para entender padrões de sucesso no cinema

In [7]:
# Carregamento dos dados
QUERY_IMDB = "SELECT * FROM IMDB_movies"
df_movies = pd.read_sql_query(QUERY_IMDB, engine)

print(f"Total de filmes: {len(df_movies)}")
print(f"\nColunas disponíveis: {', '.join(df_movies.columns)}")
print(f"\nPeríodo: {df_movies['Year'].min()} - {df_movies['Year'].max()}")

# Análise de dados faltantes
print("\nDados faltantes:")
missing = df_movies.isnull().sum()
print(missing[missing > 0])

Total de filmes: 1000

Colunas disponíveis: Id, Title, Genre, Director, Actors, Year, Runtime, Rating, Votes, RevenueMillions, Metascore

Período: 2006 - 2016

Dados faltantes:
RevenueMillions    128
Metascore           64
dtype: int64


In [8]:
def create_movie_analysis_chart(
    df: pd.DataFrame,
    top_n: int = 30,
    sort_by: str = 'RevenueMillions'
) -> None:
    """
    Cria gráfico interativo de análise de filmes.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame com dados dos filmes
    top_n : int
        Número de filmes a exibir
    sort_by : str
        Coluna para ordenação
    """
    # Preparação dos dados
    df_plot = (
        df
        .dropna(subset=['RevenueMillions', 'Rating', 'Votes'])
        .sort_values(by=sort_by, ascending=False)
        .head(top_n)
    )
    
    # Criação do gráfico
    fig = px.scatter(
        df_plot,
        x='Rating',
        y='RevenueMillions',
        size='Votes',
        color='RevenueMillions',
        hover_data=['Title', 'Genre', 'Director', 'Year'],
        title='Análise de Filmes: Crítica vs Sucesso Comercial',
        labels={
            'Rating': 'Avaliação Média (IMDB)',
            'RevenueMillions': 'Receita (Milhões USD)'
        },
        width=1200,
        height=800,
        color_continuous_scale='Viridis'
    )
    
    # Customização do layout
    fig.update_layout(
        title={'x': 0.5, 'xanchor': 'center'},
        font=dict(size=12),
        hovermode='closest'
    )
    
    fig.update_traces(
        marker=dict(
            sizemode='area',
            sizeref=2.*max(df_plot['Votes'])/(50.**2),
            line=dict(width=1, color='white')
        )
    )
    
    fig.show()
    
    # Estatísticas descritivas
    print("\n" + "="*60)
    print("ESTATÍSTICAS DOS FILMES ANALISADOS")
    print("="*60)
    print(f"\nReceita Média: ${df_plot['RevenueMillions'].mean():.2f}M")
    print(f"Avaliação Média: {df_plot['Rating'].mean():.2f}")
    print(f"Total de Votos: {df_plot['Votes'].sum():,}")
    
    # Identificação de outliers interessantes
    print("\n" + "-"*60)
    print("DESTAQUES")
    print("-"*60)
    
    high_rating = df_plot[df_plot['Rating'] >= 8.0]
    if not high_rating.empty:
        print(f"\n✓ Filmes com avaliação ≥ 8.0: {len(high_rating)}")
        print(high_rating[['Title', 'Rating', 'RevenueMillions']].to_string(index=False))
    
    high_revenue = df_plot[df_plot['RevenueMillions'] >= 500]
    if not high_revenue.empty:
        print(f"\n✓ Filmes com receita ≥ $500M: {len(high_revenue)}")
        print(high_revenue[['Title', 'Rating', 'RevenueMillions']].to_string(index=False))


# Execução da visualização
create_movie_analysis_chart(df_movies, top_n=30)


ESTATÍSTICAS DOS FILMES ANALISADOS

Receita Média: $444.17M
Avaliação Média: 7.43
Total de Votos: 15,745,796

------------------------------------------------------------
DESTAQUES
------------------------------------------------------------

✓ Filmes com avaliação ≥ 8.0: 15
                                       Title  Rating  RevenueMillions
  Star Wars: Episode VII - The Force Awakens     8.0            937.0
                                      Avatar     8.0            761.0
                                The Avengers     8.0            623.0
                             The Dark Knight     9.0            533.0
                                   Rogue One     8.0            532.0
                       The Dark Knight Rises     9.0            448.0
             The Hunger Games: Catching Fire     8.0            425.0
                                 Toy Story 3     8.0            415.0
                  Captain America: Civil War     8.0            408.0
                       

### Insights da Análise

#### O que o gráfico revela:

1. **Correlação Fraca**: Não há correlação forte entre avaliação crítica e sucesso comercial
   - Filmes com rating 7.0-8.0 podem ter receitas muito variadas
   
2. **Blockbusters de Franquia**: Filmes de grandes franquias (Star Wars, Marvel, Jurassic) dominam a receita independente da crítica

3. **Poder da Audiência**: O tamanho dos pontos (votos) indica filmes que geraram maior engajamento

4. **Oportunidades**: Filmes com alta avaliação mas baixa receita podem indicar problemas de marketing/distribuição

#### Aplicações Práticas:
- Estúdios: Identificar estratégias de sucesso
- Investidores: Avaliar potencial de ROI
- Roteiristas/Diretores: Entender trade-offs crítica vs público

## 5. Conclusão

### Resumo das Entregas

1. ✓ **Função Dinâmica de Consulta**
   - Parametrizada e segura contra SQL injection
   - Validação robusta de entrada
   - Documentação completa
   - Tratamento de erros

2. ✓ **Visualização de Ticket Médio**
   - Queries originais preservadas
   - Processamento modular e reutilizável
   - Dados agregados corretamente

3. ✓ **Análise IMDB**
   - Visualização interativa e informativa
   - Insights de negócio relevantes
   - Estatísticas descritivas complementares

### Melhorias Implementadas

- 🔧 **Organização**: Código estruturado em funções com responsabilidades claras
- 📝 **Documentação**: Docstrings detalhadas com exemplos
- 🛡️ **Segurança**: Queries parametrizadas
- ✅ **Validação**: Verificação de entrada em todas as funções
- 🎯 **Reusabilidade**: Funções genéricas e configuráveis
- 📊 **Visualização**: Gráfico interativo com múltiplas dimensões